# IN5: a steady source through the chopper train

IN5, the ILL's cold time-of-flight spectrometer, as McStas's `ILL_IN5.instr` models it. The
source is the instr's `Source_gen`: three Maxwellians standing in for the reactor's cold
source at the end of the shared H16 guide, as neutrons per second towards its target over
(time at the source, wavelength). A reactor's source is steady, so every time is alike and
the pulses are the choppers' own: the time axis's zero is arbitrary, and the source window
spans two of the train's periods by default, so that what reaches the sample is its
time-averaged rate. chopcal transmits that window through IN5's six discs exactly, as
convex polygons, and weights it by how likely each neutron is to get through while a
disc's edge is still sweeping the beam.

The discs have two slits each, at the instr's distances, radii, slit widths and heights.
Discs 1-2 make the pulse and 5-6 select the wavelength, each a pair 2 cm apart; disc 3
turns at *disc 3 ratio* of the speed and passes one pulse in every 1/ratio, and disc 4
removes what the pairs would let through at other wavelengths. Each delay is the instr's:
the time a neutron of the chosen wavelength takes from the source to the disc, so that one
leaving at zero meets every slit centred.

*counter-rotating pairs*, on by default, turns the second disc of each pair the other way,
as every description of IN5 since its upgrade around 2000 has them. The instr turns all
six the same way; turn it off for that. Counter-rotation moves no opening, so the
transmitted region is the same, but the pair's edges cross the beam in opposite
directions, which roughly halves the wavelength spread and the flux.

The band holding the chosen wavelength is intended, outlined in red; anything passing at
another wavelength is spurious, outlined and plotted in purple.

*guide transmission*, on by default, weights what reaches the sample by the fraction of
what the source emits that reaches a 10 × 25 mm window at the sample -- the instr's
sample. It comes from a McStas scan of the instr without its choppers, sample and
detectors, 10⁶ rays per point from 1 to 20 Å, so it includes what misses the guide's
30 mm entrance.

Against McStas at 4.5 Å, co-rotating, the discs pass 6.91 × 10⁶ n/s to that window here and
6.61 × 10⁶ ± 1.9 % there, with the instr's time-origin disc opened and spread over two
frames. As the instr stands it reports twice that: its `Chopper0` (`isfirst`) never
simulates the frame disc 3 closes.

Needs `chopcal>=0.8`, `ipympl` and `ipywidgets`.

In [ ]:
%matplotlib widget

from phasespace import figure
from in5 import IN5

In [ ]:
figure(IN5, wavelength=4.5, speed=8500, ratio=0.5, counter=True)

## The guide's transmission

The fraction of what IN5's `Source_gen` emits that reaches a 10 × 25 mm window at the
sample, against wavelength: the table the *guide transmission* toggle weights the source
by. From a McStas scan of `ILL_IN5.instr` with its choppers, sample and detectors removed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from in5 import IN5

wavelength, fraction, error = np.loadtxt(IN5.guide_table, delimiter=',', skiprows=1).T
fig, ax = plt.subplots(figsize=(7, 3.5), layout='constrained')
ax.errorbar(wavelength, 100 * fraction, 100 * error, fmt='.-', markersize=3)
ax.set_xlabel('wavelength [Å]')
ax.set_ylabel('reaches the sample [%]')
ax.set_ylim(bottom=0)
plt.show()

## Without widgets

`transmission` computes one setting and `draw` puts it on a figure, for a script or a
saved picture: here counter-rotating, then as the instr turns the discs.

In [ ]:
from phasespace import draw, make_axes, transmission
from in5 import IN5

for counter in (True, False):
    result = transmission(IN5, dict(wavelength=4.5, counter=counter))
    fig, axes = make_axes()
    draw(result, axes, title='counter-rotating' if counter else 'co-rotating, as the instr',
         guide=True)
    print(f'{"counter" if counter else "co"}-rotating: {result.transmitted_guide:.3g} n/s '
          'reach the sample through the guide')